1D XXZ+X experiment setup.


In [ ]:
from qiskit import *
import torch
import numpy as np
import matplotlib.pyplot as plt
import quimb as qu
from NNVQE_HEA import *
import random

torch.manual_seed(1)
np.random.seed(1)
random.seed(1)

In [ ]:
def mse(vector1, vector2):
    return np.mean((np.array(vector1) - np.array(vector2)) ** 2)


In [ ]:
sum_of_energy = []
def train(n, d, train_edge_full_data, train_node_full_data, train_latent_data, num_data, latent_size, NN_shape, maxiter=1000, lr=0.001, stddev=0.1, dropout_rate=0.1):
    model = NN_MERA_Model( n, d, stddev, NN_shape, latent_size, dropout_rate)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    batch_size = num_data
    lst = list(range(num_data))      
    
    for i in range(1, maxiter + 1):
        
        total_energy = 0.0
        for start in range(0, num_data, batch_size):
            chunk = lst[start : start + batch_size]   # take a batch slice
            batch_energy = torch.tensor(0.0, dtype=torch.float32)
            optimizer.zero_grad()
            for j in chunk:
                node_feats = train_node_full_data[j]
                edge_data = [t.tolist() for t in train_edge_full_data[j].values()]
                edges = list(train_edge_full_data[j].keys())
                graph_latent = train_latent_data[j]
                energy,_ = model(edges, edge_data, node_feats, graph_latent)
                batch_energy += energy
                total_energy += energy.item()
            # total_energy = total_energy.to(torch.float32)
            batch_energy.backward()
            optimizer.step()
        # scheduler.step()
        sum_of_energy.append(total_energy)
        if i % 10 == 0:
            print(f"Epoch {i}, Total Energy: {total_energy}")
    
        torch.save(model.state_dict(), "EGATE_NNVQE_HEA_1layer.pth")


In [ ]:
n = 8
d = 2
#  [t.tolist() for t in edge_data[1].values()]
cirq, idx = HEA({"params": np.zeros(1000),  "edges": None, "edge_data":None, "node_feats":None}, n, d, param_num=True)
print("The number of parameters is", idx)
# cirq.draw("mpl")

In [ ]:
train_edge_full_data = torch.load('edge_full_data_train.pt')
train_latent_data = torch.load('latent_data_train.pt')
train_node_full_data = torch.load('node_full_data_train.pt')

In [ ]:

n = 8
depth = 2

latent_size = len(train_latent_data[0])   
NN_shape = 20
stddev = 0.1
maxiter = 200
lr = 0.003
stddev = 0.1
dropout_rate=0.05

train(n, depth, train_edge_full_data, train_node_full_data, train_latent_data, len(train_latent_data), latent_size, NN_shape, maxiter, lr, stddev,dropout_rate)

In [ ]:
X = range(maxiter)
Y = []
for i in range(maxiter):
    Y.append(sum_of_energy[i])
plt.plot(X, Y)
plt.xlabel("# of epoch")
plt.ylabel("cost") #, fontsize=14
plt.show()

In [ ]:
print("GAE_NNVQE_sum_of_energy_1 = ",sum_of_energy)

## Evaluation


In [ ]:
def test(n, d, test_edge_full_data,test_node_full_data,test_latent_data, num, latent_size, NN_shape, stddev=0.1, dropout_rate=0.1):
  test_energy = []
  final_state = []
  model = NN_MERA_Model( n, d, stddev, NN_shape, latent_size, dropout_rate)
  model.load_state_dict(torch.load("EGATE_NNVQE_HEA_1layer.pth"))
  model.eval()
  for j in range(num):
    node_feats = test_node_full_data[j]
    edge_data = [t.tolist() for t in test_edge_full_data[j].values()]
    edges = list(test_edge_full_data[j].keys())
    graph_latent = test_latent_data[j]
    energy, state = model(edges, edge_data, node_feats, graph_latent, compute_state= True)
    test_energy.append(energy.item())
    final_state.append(state)
  return test_energy, final_state

In-distribution test data.


In [ ]:
# analytical_energies =  analytical_energy(n, train_edge_full_data, train_node_full_data, len(train_latent_data))
analytical_energies = torch.load('analytical_energies_train.pt')
print("total_analytical_energies_sum = ", np.array(analytical_energies).sum())
print("analytical_energies_train = ",analytical_energies)


In [ ]:
# analytical_states =  analytical_state(n, train_edge_full_data, train_node_full_data, len(train_latent_data))
analytical_states = torch.load('analytical_states_train.pt')


In [ ]:
train_test_energy, final_state = test(n, depth,train_edge_full_data, train_node_full_data, train_latent_data, len(train_latent_data), latent_size, NN_shape, stddev, dropout_rate)

In [ ]:
def mse(vector1, vector2):
    return np.mean((np.array(vector1) - np.array(vector2)) ** 2)


In [ ]:

def calculate_fidelity(psi, phi):
    # Normalize
    phi = phi.detach().numpy()
    psi = psi / np.linalg.norm(psi)
    phi = phi / np.linalg.norm(phi)
    
    # Align global phase
    phase = np.vdot(psi, phi) / abs(np.vdot(psi, phi))
    phi_aligned = phi / phase
    
    # Compute fidelity
    fidelity = abs(np.vdot(psi, phi_aligned))**2
    return fidelity

def fidelity_torch(psi, phi, eps=1e-12):
    psi = torch.from_numpy(psi)
    # phi = phi.detach().numpy()
    psi = psi / torch.linalg.vector_norm(psi)  # handles both complex and real tensors
    phi = phi / torch.linalg.vector_norm(phi)

    return torch.abs(torch.vdot(psi.conj(), phi))**2 

def calculate_dataset_fidelities(states_A, states_B):
    assert len(states_A) == len(states_B), "Datasets must have the same number of states."
    
    fidelities = [fidelity_torch(psi, phi) for psi, phi in zip(states_A, states_B)]

    return fidelities



In [ ]:
fidelity = calculate_dataset_fidelities(analytical_states, final_state)

fidelities = []

for i in range(len(fidelity)):
    fidelities.append(fidelity[i].item())

fidelities = np.array(fidelities)
# relative error
favg = np.mean(fidelities)
plt.plot(
    list(range(len(train_latent_data))), fidelities, "-", color="b",linewidth= 1
)
plt.ylabel("Fidelity", fontsize=14)
plt.axhline(favg,linestyle='--', linewidth= 1, label = f"mean = {favg:.5f}",  color = 'red')
plt.legend()
plt.show()

In [ ]:
# relative error
err = [a - b for a, b in zip(train_test_energy, analytical_energies)]/ np.abs(analytical_energies)
avg = np.mean(err)
plt.plot(
    list(range(len(train_latent_data))), err, "-", color="b",linewidth= 1
)
plt.xlabel(r"$\lambda$", fontsize=14)
plt.ylabel("GS Relative Error", fontsize=14)
plt.axhline(avg,linestyle='--', linewidth= 1, label = f"mean = {avg:.5f}",  color = 'red')
plt.legend()
plt.show()

In [ ]:
mse_value = mse(train_test_energy, analytical_energies)
print(mse_value, avg, favg)

In [ ]:
print("GAE_NNVQE_train_test_energies_1 = ",train_test_energy)

Out-of-range test data.


In [ ]:
test_edge_full_data = torch.load('edge_full_data_test.pt')
test_node_full_data = torch.load('node_full_data_test.pt')
test_latent_data = torch.load('latent_data_test.pt')

In [ ]:
# analytical_energies =  analytical_energy(n,test_edge_full_data, test_node_full_data, len(test_latent_data))
analytical_energies = torch.load('analytical_energies_test.pt')
# analytical_states =  analytical_state(n,test_edge_full_data, test_node_full_data, len(test_latent_data))
analytical_states = torch.load('analytical_states_test.pt')
print("analytical_energies_test = ",analytical_energies)
test_energy, final_state = test(n, depth, test_edge_full_data, test_node_full_data, test_latent_data, len(test_latent_data), latent_size, NN_shape, stddev, dropout_rate)
# relative error

fidelity = calculate_dataset_fidelities(analytical_states, final_state)

fidelities = []

for i in range(len(fidelity)):
    fidelities.append(fidelity[i].item())

fidelities = np.array(fidelities)
# relative error
favg = np.mean(fidelities)
plt.plot(
    list(range(len(test_latent_data))), fidelities, "-", color="b",linewidth= 1
)
plt.ylabel("Fidelity", fontsize=14)
plt.axhline(favg,linestyle='--', linewidth= 1, label = f"mean = {favg:.5f}",  color = 'red')
plt.legend()
plt.show()


In [ ]:
# relative error
err = [a - b for a, b in zip(test_energy, analytical_energies)]/ np.abs(analytical_energies)
avg = np.mean(err)
plt.plot(
    list(range(len(test_latent_data))), err, "-", color="b",linewidth= 1
)
plt.xlabel(r"$\lambda$", fontsize=14)
plt.ylabel("GS Relative Error", fontsize=14)
plt.axhline(avg,linestyle='--', linewidth= 1, label = f"mean = {avg:.5f}",  color = 'red')
plt.legend()
plt.show()

In [ ]:
mse_value = mse(test_energy, analytical_energies)
print("prop = ",[mse_value, avg, favg])

In [ ]:
print("GAE_NNVQE_test_energies_1 = ",test_energy)